# From NumPy to PyTorch

**Goal.** Write the same layer, `x @ W + b`, in PyTorch. The axes mean what they meant on day 1.

**Objective.** Given a tensor that is already familiar from NumPy, write the PyTorch version and name what changed: `dtype`, `dim`, `permute`, `view`, and `.grad`.


Day 1 settled what a product means. This notebook changes the spelling.

`shape`, `ndim`, `@`, `*`, indexing, `reshape`, and broadcasting from the right all stay. A leading axis is still the arrangement. The shared length is still used up.

Five things change.

- `torch.tensor` replaces `np.array`. A Python list of decimals becomes `float32`. NumPy used `float64`.
- `dim` replaces `axis`.
- `permute` reorders every axis. `transpose` swaps exactly two. NumPy's `transpose(1, 0, 2)` is `permute(1, 0, 2)` here.
- `view` regroups a tensor only when the entries already sit in one contiguous block. `reshape` still works when a copy is required.
- `requires_grad=True` and `backward()` store a gradient on `.grad`. NumPy has no such slot.

A failed shape raises `RuntimeError`, where NumPy raised `ValueError`.


In [2]:
import numpy as np
import torch

torch.set_printoptions(sci_mode=False)

def describe(name, tensor):
    print(f'{name}: shape={tuple(tensor.shape)}, ndim={tensor.ndim}, dtype={tensor.dtype}')
    print(tensor)
    print("\n")


## 1. Building the tensor

`torch.tensor` on a Python list of decimals gives `float32`. The same numbers in NumPy were `float64`. The device here is `cpu`.

`torch.tensor` on a NumPy array copies the values and keeps that array's dtype, so a `float64` array stays `float64`. `torch.from_numpy` does not copy. Change the array and the tensor changes with it.

A 0-dim tensor is still a tensor. `.item()` pulls the Python number out of it.


In [15]:
scalar = torch.tensor(3.0)                 # (), float32
vector = torch.tensor([1.0, 2.0, 3.0])     # (3,), float32
print('device:', scalar.device)

numpy_vector = np.array([1.0, 2.0, 3.0])   # float64
copied = torch.tensor(numpy_vector)        # copy, dtype stays float64
shared = torch.from_numpy(numpy_vector)    # same memory, float64

describe('from a list', vector)
describe('copied from NumPy', copied)
describe('shared with NumPy', shared)

numpy_vector[0] = 9.0
describe('shared after the array changes', shared)
describe('copy after the array changes', copied)

entry = vector[0]
print('entry:', entry, 'item:', entry.item())

assert scalar.dtype == torch.float32
assert scalar.device.type == 'cpu'
assert vector.dtype == torch.float32
assert copied.dtype == torch.float64
assert shared.dtype == torch.float64
assert shared[0].item() == 9.0
assert copied[0].item() == 1.0
assert entry.ndim == 0
assert entry.item() == 1.0


device: cpu
from a list: shape=(3,), ndim=1, dtype=torch.float32
tensor([1., 2., 3.])


copied from NumPy: shape=(3,), ndim=1, dtype=torch.float64
tensor([1., 2., 3.], dtype=torch.float64)


shared with NumPy: shape=(3,), ndim=1, dtype=torch.float64
tensor([1., 2., 3.], dtype=torch.float64)


shared after the array changes: shape=(3,), ndim=1, dtype=torch.float64
tensor([9., 2., 3.], dtype=torch.float64)


copy after the array changes: shape=(3,), ndim=1, dtype=torch.float64
tensor([1., 2., 3.], dtype=torch.float64)


entry: tensor(1.) item: 1.0


## 2. `dim` is the old `axis`

`sum` collapses one axis. `dim=0` sums down the rows. `dim=1` sums across each row. The call in NumPy was `matrix.sum(axis=0)`.

`torch.stack(..., dim=0)` puts the new axis in front. `dim=2` puts it at the end. `vstack` and `hstack` keep their NumPy names and their shapes: `(4, 2)` and `(2, 4)`.


In [ ]:
matrix = torch.tensor([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0],
])  # (2, 3)

describe('sum dim 0', matrix.sum(dim=0))  # [5, 7, 9]
describe('sum dim 1', matrix.sum(dim=1))  # [6, 15]
assert torch.allclose(matrix.sum(dim=0), torch.tensor([5.0, 7.0, 9.0]))
assert torch.allclose(matrix.sum(dim=1), torch.tensor([6.0, 15.0]))

square = torch.tensor([
    [1.0, 0.0],
    [0.0, 1.0],
])  # (2, 2)

front = torch.stack([square, square], dim=0)  # (2, 2, 2)
back = torch.stack([square, square], dim=2)   # (2, 2, 2)
under = torch.vstack([square, square])        # (4, 2)
beside = torch.hstack([square, square])       # (2, 4)

assert tuple(front.shape) == (2, 2, 2)
assert torch.equal(front[0], square)
assert torch.equal(back[:, :, 0], square)
assert tuple(under.shape) == (4, 2)
assert tuple(beside.shape) == (2, 4)


## 3. The same layer

`@` and `+` are unchanged. `(3, 2) @ (2, 4) + (4,)` is still `(3, 4)`. The first row `[2, -1]` still becomes `[2.1, -0.8, -2.7, 5.4]`.

A leading axis still stays: `(2, 3, 2) @ (2, 4) + (4,)` is `(2, 3, 4)`.


In [ ]:
x = torch.tensor([
    [2.0, -1.0],
    [0.5, 3.0],
    [-2.0, 1.0],
])  # (3, 2)

W = torch.tensor([
    [1.0, 0.0, -1.0, 2.0],
    [0.0, 1.0,  1.0, -1.0],
])  # (2, 4)

b = torch.tensor([0.1, 0.2, 0.3, 0.4])  # (4,)

y = x @ W + b  # (3, 4)
describe('y', y)
assert tuple(y.shape) == (3, 4)
assert torch.allclose(y[0], torch.tensor([2.1, -0.8, -2.7, 5.4]))

x_batch = torch.stack([
    x,
    torch.tensor([
        [1.0, 1.0],
        [0.0, 2.0],
        [4.0, -1.0],
    ]),
])  # (2, 3, 2)

y_batch = x_batch @ W + b  # (2, 3, 4)
assert tuple(y_batch.shape) == (2, 3, 4)
assert torch.allclose(y_batch[0], y)


## 4. `reshape` and `view`

`reshape` is the same regroup as in NumPy. Six entries become `(2, 3)`. `reshape(2, 4)` fails, because `2 × 4` is 8.

On a fresh tensor the entries are contiguous, so `view(2, 3)` returns the same groups. `view` is the call that refuses to copy. The next section builds a tensor where it refuses.


In [ ]:
row = torch.tensor([1, 2, 3, 4, 5, 6])  # (6,)
groups = row.reshape(2, 3)
same = row.view(2, 3)

describe('groups', groups)
assert torch.equal(groups, torch.tensor([[1, 2, 3], [4, 5, 6]]))
assert torch.equal(same, groups)

try:
    row.reshape(2, 4)
except RuntimeError as error:
    print(f'reshape(2, 4): {type(error).__name__}: {error}')
else:
    raise AssertionError('6 numbers should not fit the shape (2, 4)')


## 5. `permute` replaces a full transpose

`.T` still swaps the two axes of a matrix. `[1, 2, 3]` becomes the first column.

For three axes, NumPy took the whole order in one call: `transpose(1, 0, 2)`. PyTorch splits that into two names. `transpose(0, 1)` swaps those two axes and leaves the last one in place. `permute(1, 0, 2)` names the full order: axis 1, then axis 0, then axis 2.

The head split from day 1 is `(2, 2, 2)`, in the order rows, heads, numbers. Both calls put heads first, so head 0 is

```text
[[1, 2],
 [5, 6]]
```

`permute` changes the stride. The entries are no longer one contiguous block, so `view(4, 2)` raises `RuntimeError`. `reshape(4, 2)` still reads them, head 0 and then head 1.


In [ ]:
matrix = torch.tensor([
    [1, 2, 3],
    [4, 5, 6],
])  # (2, 3)
assert tuple(matrix.T.shape) == (3, 2)
assert torch.equal(matrix.T[:, 0], torch.tensor([1, 2, 3]))

features = torch.tensor([
    [1, 2, 3, 4],
    [5, 6, 7, 8],
])  # (2, 4)
split = features.reshape(2, 2, 2)  # (rows, heads, numbers)

by_head = split.permute(1, 0, 2)
by_swap = split.transpose(0, 1)
describe('by_head', by_head)

assert torch.equal(by_head, by_swap)
assert torch.equal(by_head[0], torch.tensor([[1, 2], [5, 6]]))

try:
    by_head.view(4, 2)
except RuntimeError as error:
    print(f'view: {type(error).__name__}: {error}')
else:
    raise AssertionError('view should refuse a non-contiguous tensor')

regrouped = by_head.reshape(4, 2)
describe('regrouped', regrouped)
assert torch.equal(regrouped, torch.tensor([[1, 2], [5, 6], [3, 4], [7, 8]]))


## 6. Broadcasting did not change

Shapes still line up from the right. A size fits when it equals the other size, or when it is 1. A missing size on the left counts as 1.

```text
(2, 3) + (3,)   copies [0.1, 0.2, 0.3] down both rows
(2, 3) + (2, 1) copies 10 across the first row and 20 across the second
(2, 3) + (2,)   2 against 3, so RuntimeError
```


In [ ]:
rows = torch.tensor([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0],
])  # (2, 3)

added_columns = rows + torch.tensor([0.1, 0.2, 0.3])  # (3,)
added_rows = rows + torch.tensor([[10.0], [20.0]])    # (2, 1)

describe('added_columns', added_columns)
describe('added_rows', added_rows)
assert torch.allclose(added_columns, torch.tensor([[1.1, 2.2, 3.3], [4.1, 5.2, 6.3]]))
assert torch.allclose(added_rows, torch.tensor([[11.0, 12.0, 13.0], [24.0, 25.0, 26.0]]))

try:
    rows + torch.tensor([0.1, 0.2])
except RuntimeError as error:
    print(f'length 2: {type(error).__name__}: {error}')
else:
    raise AssertionError('a vector of length 2 should not add to rows of length 3')


## 7. A gradient on the same layer

`requires_grad=True` marks `W` and `b`. The line `y = x @ W + b` is remembered. `y.sum()` adds every entry into one number, and `backward()` writes one slope per marked entry.

`W.grad` has shape `(2, 4)`, the same shape as `W`. The first column of `x` sums to `0.5`, so the first row of `W.grad` is `[0.5, 0.5, 0.5, 0.5]`. The second column of `x` sums to `3`, so the second row is all `3`s. `b` is added once per row, and there are 3 rows, so `b.grad` is `[3, 3, 3, 3]`.

`x` was not marked, so `x.grad` is `None`.


In [14]:
x = torch.tensor([
    [2.0, -1.0],
    [0.5, 3.0],
    [-2.0, 1.0],
])  # (3, 2)

W = torch.tensor([
    [1.0, 0.0, -1.0, 2.0],
    [0.0, 1.0,  1.0, -1.0],
], requires_grad=True)  # (2, 4)

b = torch.tensor([0.1, 0.2, 0.3, 0.4], requires_grad=True)  # (4,)

y = x @ W + b
y.sum().backward()

describe('W.grad', W.grad)
describe('b.grad', b.grad)
print('x.grad:', x.grad)

assert tuple(W.grad.shape) == (2, 4)
assert tuple(b.grad.shape) == (4,)
assert torch.allclose(W.grad, torch.tensor([
    [0.5, 0.5, 0.5, 0.5],
    [3.0, 3.0, 3.0, 3.0],
]))
assert torch.allclose(b.grad, torch.tensor([3.0, 3.0, 3.0, 3.0]))
assert x.grad is None


W.grad: shape=(2, 4), ndim=2, dtype=torch.float32
tensor([[0.5000, 0.5000, 0.5000, 0.5000],
        [3.0000, 3.0000, 3.0000, 3.0000]])


b.grad: shape=(4,), ndim=1, dtype=torch.float32
tensor([3., 3., 3., 3.])


x.grad: None


## 8. Checkpoint

Answer these from the names. Do not run them.

1. What is the dtype of `torch.tensor([1.0, 2.0])`? What is the dtype of `torch.tensor(np.array([1.0, 2.0]))`?
2. `matrix` has shape `(2, 3)`. What is `matrix.sum(dim=0).shape`?
3. A tensor has shape `(2, 3, 4)`. What is `permute(2, 0, 1).shape`? What is `transpose(0, 2).shape`?
4. After `permute`, `view` refuses. Which call still regroups the entries?
5. `W` has shape `(2, 4)` and `requires_grad=True`. After `y.sum().backward()`, what is `W.grad.shape`?

<details>
<summary>Answers</summary>

1. The list becomes `torch.float32`. The NumPy array is `float64`, and `torch.tensor` keeps that dtype.

2. `(3,)`. `dim=0` sums down the two rows and leaves the three columns.

3. `permute(2, 0, 1)` takes axis 2, then 0, then 1, so the shape is `(4, 2, 3)`. `transpose(0, 2)` swaps the first axis and the last, so the shape is `(4, 3, 2)`.

4. `reshape`. `view` requires one contiguous block.

5. `(2, 4)`. One slope per entry of `W`.

</details>


## What this establishes

The layer is still `x @ W + b`, read from the right. The spelling around it is now PyTorch's.

Decimals written as a list are `float32`. An axis argument is `dim`. A full reorder is `permute`, and `transpose` swaps two axes. `view` regroups a contiguous tensor. `reshape` works when the entries have been reordered. `backward()` fills `.grad` with a tensor of the same shape as the marked parameter.

The two questions from day 1 are unchanged: what each axis means, and what shape must come out.
